# 04 — Direct 1+1D vs 3D validation

Runs individual BubbleMaster 1+1D simulations for a small set of
(lambda_bar, gamma) points and compares the resulting GW spectrum
directly against the corresponding sledgehamr 3D output.

**Workflow**
1. Fill in `VALIDATION_POINTS` with your (lambda_bar, gamma, sledgehamr_path) triples.
2. Run **Section 1** to write setup HDF5 files and SLURM scripts.
3. Submit jobs: `! sbatch scripts/validate_<tag>.sh`
4. After jobs complete, run **Section 2** onward to load and compare.

In [ ]:
import sys
import numpy as np
import h5py
import matplotlib.pyplot as plt
import ptbuilder as pt
from pathlib import Path
from ptbuilder.ic import _two_bubble_ic, _cutoff_times, BubbleMasterParams

# Bubble locations in sledgehamr reference lambda_bar=0.84
xlocs = [7.225e+1, 1.278e+2, 1.087e+2]
ylocs = [8.992e+1, 8.992e+1, 1.101e+2]
zlocs = [1.000e+2, 1.000e+2, 1.000e+2]
pos = np.column_stack((xlocs, ylocs, zlocs))
distances = np.linalg.norm(pos[:, None, :] - pos[None, :, :], axis=-1)
distances = np.unique(distances)
print(distances)

# slightly different rounding for the special gamma_2 run
distance_gamma2 = 1.139e+2 - 8.615e+1


# Bubble locations in sledgehamr reference lambda_bar=0.069
xlocs = [1.623e+1, 2.877e+1, 2.408e+1]
ylocs = [1.906e+1, 1.906e+1, 2.594e+1]
zlocs = [2.250e+1, 2.250e+1, 2.250e+1]
pos = np.column_stack((xlocs, ylocs, zlocs))
distances_2 = np.linalg.norm(pos[:, None, :] - pos[None, :, :], axis=-1)
distances_2 = np.unique(distances_2)
print(distances_2)

In [ ]:
# Repo root — all generated files go here, not in notebooks/
REPO_ROOT = Path(pt.__file__).parent.parent

SH_RUNS_DIR  = Path('/pscratch/sd/b/buschman/other_runs/')
BM_SCAN_BASE = REPO_ROOT / 'data'

# Box parameters for sledgehamr runs (must match what was simulated)
SH_L        = [200., 200., 200., 50., 50., 50.]
SH_ZERO_PAD = [2., 2., 2., 2., 2., 2.]

# Scan time and gamma grids (must match what was run on Perlmutter)
SCAN_TIMES  = np.linspace(1., 70., 32)
SCAN_GAMMAS = np.linspace(1.,  8., 32)

VALIDATION_POINTS = [
    dict(lambda_bar=0.84,  d=distance_gamma2, gamma=1.937051,
         sh_path=SH_RUNS_DIR / 'bubble_normalization_gamma2'),
    dict(lambda_bar=0.84,  d=distances[2],    gamma=2.962170,
         sh_path=SH_RUNS_DIR / 'bubble_N3_234_0_2_'),
    dict(lambda_bar=0.84,  d=distances[3],    gamma=3.973938,
         sh_path=SH_RUNS_DIR / 'bubble_N3_234_0_1_'),
    dict(lambda_bar=0.069, d=distances_2[1],  gamma=3.872952,
         sh_path=SH_RUNS_DIR / 'bubble_N3_456_1_2_'),
    dict(lambda_bar=0.069, d=distances_2[2],  gamma=4.900938,
         sh_path=SH_RUNS_DIR / 'bubble_N3_456_0_2_'),
    dict(lambda_bar=0.069, d=distances_2[3],  gamma=5.916847,
         sh_path=SH_RUNS_DIR / 'bubble_N3_456_0_1_'),
]

config = pt.Config()
params = BubbleMasterParams(dz=0.005, n_w=64, n_k=51,
                             how_often_ds=5, baby_steps=100, cutoff_type=0)

## 1. Write setup files and SLURM scripts

In [ ]:
scripts_dir = REPO_ROOT / 'scripts'
logs_dir    = REPO_ROOT / 'logs'
scripts_dir.mkdir(exist_ok=True)
logs_dir.mkdir(exist_ok=True)

for vp in VALIDATION_POINTS:
    lb = vp['lambda_bar']
    d  = vp['d']

    model   = pt.PhysicsModel(pt.Phi4Potential(lb), config)
    gamma   = gamma_from_d(model, d)
    vp['gamma'] = gamma   # store for later use

    tag = f'lb{lb}_g{gamma:.4g}'

    # Time grid derived from physical scales
    t_0, t_cut, t_m, t_max = _cutoff_times(d, params.cutoff_type, params.t_0_scal)
    times = np.linspace(t_m - 2*t_0, t_m + 2*t_0, 5)

    run_dir    = model.results_dir / 'validation' / tag
    setup_path = run_dir / 'setup.h5'
    out_dir    = run_dir / 'out'
    run_dir.mkdir(parents=True, exist_ok=True)

    pt.write_2d_setup(model, gamma=gamma, times=times,
                      path=setup_path, params=params)

    print(f'[{tag}]')
    print(f'  d={d:.4f}  gamma(wall-thinning)={gamma:.6f}')
    print(f'  t_cut={t_cut:.3f}  t_max={t_max:.3f}')
    print(f'  setup -> {setup_path}')

    # --- CPU (GSL) SLURM script ---
    cpu_script_path = scripts_dir / f'validate_{tag}.sh'
    cpu_header = (
        '#!/bin/bash\n'
        f'#SBATCH --job-name=bm_{tag}\n'
        f'#SBATCH --output={logs_dir}/bm_{tag}_%j.out\n'
        '#SBATCH --constraint=cpu\n'
        '#SBATCH --qos=premium\n'
        '#SBATCH --nodes=1\n'
        '#SBATCH --tasks-per-node=1\n'
        '#SBATCH --cpus-per-task=128\n'
        '#SBATCH --time=01:00:00\n'
    )
    cpu_run_cmd = (
        'export SLURM_CPU_BIND=cores\n'
        'export OMP_PLACES=threads\n'
        'export OMP_PROC_BIND=spread\n'
        'export OMP_NUM_THREADS=128\n'
        f'mkdir -p {out_dir}\n'
        f'/usr/bin/time --verbose srun {config.bubblemaster_bin} '
        f'{setup_path} {out_dir}/ --save-fields\n'
    )
    cpu_script_path.write_text(cpu_header + '\n' + cpu_run_cmd)

    # --- GPU SLURM script ---
    gpu_script_path = scripts_dir / f'validate_{tag}_gpu.sh'
    gpu_header = (
        '#!/bin/bash\n'
        f'#SBATCH --job-name=bm_{tag}_gpu\n'
        f'#SBATCH --output={logs_dir}/bm_{tag}_gpu_%j.out\n'
        '#SBATCH --constraint=gpu\n'
        '#SBATCH --qos=shared\n'
        '#SBATCH -A m3166_g\n'
        '#SBATCH --nodes=1\n'
        '#SBATCH --ntasks=1\n'
        '#SBATCH --cpus-per-task=32\n'
        '#SBATCH --gpus-per-task=1\n'
        '#SBATCH --time=02:30:00\n'
    )
    gpu_run_cmd = (
        f'mkdir -p {out_dir}\n'
        f'/usr/bin/time --verbose srun {config.bubblemaster_gpu_bin} '
        f'{setup_path} {out_dir}/ --save-fields\n'
    )
    gpu_script_path.write_text(gpu_header + '\n' + gpu_run_cmd)

    print(f'  scripts -> {cpu_script_path.name}   |   {gpu_script_path.name}')
    print()

print('Done. Both CPU and GPU scripts were written for each point — submit whichever')
print('backend you want (they write into the same out_dir/, so submit only one per point):')
print('  sbatch scripts/validate_<tag>.sh')
print('  sbatch scripts/validate_<tag>_gpu.sh')


Submit jobs from the repo root (not from notebooks/):
```bash
cd /path/to/PhaseTransitionBuilder
sbatch scripts/validate_lb0.84_g2.0.sh
sbatch scripts/validate_lb0.84_g3.0.sh
sbatch scripts/validate_lb0.84_g4.0.sh
```
---
## 2. Load BubbleMaster results (run after jobs complete)

In [ ]:
def load_bm_result(setup_path, out_dir):
    """Load BubbleMaster spectrum log-interpolated to t_m from validation run output."""
    out_dir = Path(out_dir)
    with h5py.File(setup_path, 'r') as f:
        t_m   = float(f.attrs['t_m'])
        times = f['times'][:]

    entries = []
    for p in out_dir.glob('result_*.h5'):
        i_t = int(p.stem.split('_')[1])
        if i_t < len(times):
            entries.append((times[i_t], i_t, p))
    if not entries:
        raise FileNotFoundError(f'No result_*.h5 in {out_dir}')
    entries.sort()

    t_avail = np.array([e[0] for e in entries])

    if t_m < t_avail[0] or t_m > t_avail[-1]:
        raise RuntimeError(
            f't_m={t_m:.4f} outside completed range [{t_avail[0]:.4f}, {t_avail[-1]:.4f}]'
        )

    if len(entries) == 1 or t_m == t_avail[0]:
        _, _, p = entries[0]
        with h5py.File(p, 'r') as f:
            w, spec = f['w'][:], f['spectrum'][:]
        return w, spec, t_m

    i = int(np.searchsorted(t_avail, t_m)) - 1
    i = max(0, min(i, len(entries) - 2))
    _, _, p_a = entries[i]
    _, _, p_b = entries[i + 1]
    with h5py.File(p_a, 'r') as f:
        w, spec_a = f['w'][:], f['spectrum'][:]
    with h5py.File(p_b, 'r') as f:
        spec_b = f['spectrum'][:]
    alpha = (t_m - t_avail[i]) / (t_avail[i + 1] - t_avail[i])
    floor = max(spec_a.max(), spec_b.max()) * 1e-12
    log_a = np.log(np.maximum(spec_a, floor))
    log_b = np.log(np.maximum(spec_b, floor))
    spec  = np.exp(log_a + alpha * (log_b - log_a))
    print(f'  interpolated t_m={t_m:.4f} between t={t_avail[i]:.4f} and t={t_avail[i+1]:.4f} (alpha={alpha:.3f})')
    return w, spec, t_m


bm_results = {}
for vp in VALIDATION_POINTS:
    lb    = vp['lambda_bar']
    gamma = vp['gamma']
    tag   = f'lb{lb}_g{gamma:.4g}'

    data_dir   = REPO_ROOT / 'data' / f'phi4_lambda_bar{lb}'
    setup_path = data_dir / 'validation' / tag / 'setup.h5'
    out_dir    = data_dir / 'validation' / tag / 'out'

    try:
        w, spec, t_m = load_bm_result(setup_path, out_dir)
        bm_results[tag] = dict(w=w, spec=spec, d=vp['d'], lb=lb, gamma=gamma, t_m=t_m)
        print(f'[{tag}]  d={vp["d"]:.3f}  t_m={t_m:.4f}')
    except (FileNotFoundError, RuntimeError) as e:
        print(f'[{tag}]  MISSING/INCOMPLETE: {e}')

## 3. Load sledgehamr 3D spectra

In [ ]:
import glob, os

# ---------------------------------------------------------------------------
# k_eff² correction: load precomputed register (dict keyed by box size N)
# ---------------------------------------------------------------------------
_KEFFSQ_PATH = REPO_ROOT / 'reference_codes/FirstOrder_GW/collision_weights/keff.npy'
_keffsq_register = np.load(_KEFFSQ_PATH, allow_pickle=True).item()

def _dim_from_nbins(nbins):
    dim_est = 2 * (nbins - 1) / np.sqrt(3)
    return 2 ** int(round(np.log2(dim_est)))

def _apply_keffsq(k_int):
    return _keffsq_register[_dim_from_nbins(len(k_int))]

# ---------------------------------------------------------------------------
# Direct HDF5 reader for sledgehamr gw_spectra/<i>/spectra.hdf5
# ---------------------------------------------------------------------------

def _list_gw_files(sh_path):
    pattern = os.path.join(sh_path, 'gw_spectra', '*', 'spectra.hdf5')
    entries = []
    for fp in glob.glob(pattern):
        dname = os.path.basename(os.path.dirname(fp))
        if dname.isdigit():
            entries.append((int(dname), fp))
    entries.sort(key=lambda x: x[0])
    return entries

def _read_gw_file(fp):
    with h5py.File(fp, 'r') as f:
        k_int = np.asarray(f['k' if 'k' in f else 'k_sq'][:])
        spec  = np.asarray(f['Spectrum' if 'Spectrum' in f else 'spectrum'][:])
        hdr   = np.asarray(f['Header'][:]).reshape(-1)
        t     = float(hdr[0])
    return _apply_keffsq(k_int), spec, t


def load_sledgehamr_spectrum(sh_path, t_target, L=200., zero_pad=2.):
    """
    Load the sledgehamr GW spectrum log-interpolated to time t_target.
    """
    files = _list_gw_files(sh_path)
    if not files:
        raise FileNotFoundError(f'No gw_spectra/*/spectra.hdf5 found in {sh_path}')

    snap_times = []
    for _, fp in files:
        with h5py.File(fp, 'r') as f:
            snap_times.append(float(np.asarray(f['Header'][:]).reshape(-1)[0]))
    snap_times = np.array(snap_times)

    if t_target <= snap_times[0]:
        k_sq, spec, t = _read_gw_file(files[0][1])
        print(f'  t_target={t_target:.3f} before first snapshot (t={snap_times[0]:.3f}), using first')
        return k_sq, spec, snap_times[0]

    if t_target >= snap_times[-1]:
        k_sq, spec, t = _read_gw_file(files[-1][1])
        print(f'  t_target={t_target:.3f} after last snapshot (t={snap_times[-1]:.3f}), using last')
        return k_sq, spec, snap_times[-1]

    i = int(np.searchsorted(snap_times, t_target)) - 1
    k_sq, spec_a, t_a = _read_gw_file(files[i][1])
    _,    spec_b, t_b = _read_gw_file(files[i + 1][1])
    alpha = (t_target - t_a) / (t_b - t_a)
    floor = max(spec_a.max(), spec_b.max()) * 1e-12
    log_a = np.log(np.maximum(spec_a, floor))
    log_b = np.log(np.maximum(spec_b, floor))
    spec_interp = np.exp(log_a + alpha * (log_b - log_a))
    print(f'  interpolated t={t_target:.3f} between snapshots t={t_a:.3f} and t={t_b:.3f} (alpha={alpha:.3f})')
    return k_sq, spec_interp, t_target


def normalize_sh_spectrum(k_sq, spectrum, d, L=200., zero_pad=2.):
    if k_sq[0] == 0:
        k_sq     = k_sq[1:]
        spectrum = spectrum[1:]
    Leff = L * zero_pad
    r    = np.sqrt(k_sq)
    kR   = r * (2*np.pi/Leff) * d
    kL   = 2*np.pi * r
    norm = 4.0 * kL * Leff**3
    return kR, spectrum * norm


sh_results = {}
for vp in VALIDATION_POINTS:
    lb    = vp['lambda_bar']
    gamma = vp['gamma']
    tag   = f'lb{lb}_g{gamma:.4g}'

    t_m = bm_results[tag]['t_m'] if tag in bm_results else None
    d   = bm_results[tag]['d']   if tag in bm_results else vp.get('d')

    if t_m is None:
        print(f'[{tag}]  no t_m available, skipping sledgehamr load')
        continue

    try:
        k_sq, spec, t = load_sledgehamr_spectrum(
            vp['sh_path'], t_target=t_m, L=SH_L, zero_pad=SH_ZERO_PAD)
        sh_results[tag] = dict(k_sq=k_sq, spec=spec, t=t, d=d)
        print(f'[{tag}]  sledgehamr at t={t:.3f}')
    except Exception as e:
        print(f'[{tag}]  sledgehamr not loaded: {e}')

## 4. Compare 1+1D vs 3+1D

In [ ]:
colors = ['#0072B2', '#D55E00', '#009E73', '#CC79A7', 'black', 'black']

fig, axes = plt.subplots(1, len(VALIDATION_POINTS),
                         figsize=(5 * len(VALIDATION_POINTS), 5),
                         sharey=True)
if len(VALIDATION_POINTS) == 1:
    axes = [axes]

for ax, vp, color, L, pad in zip(axes, VALIDATION_POINTS, colors, SH_L, SH_ZERO_PAD):
    lb    = vp['lambda_bar']
    gamma = vp['gamma']
    tag   = f'lb{lb}_g{gamma:.4g}'

    # 1+1D
    if tag in bm_results:
        r  = bm_results[tag]
        kR = r['w'] * r['d']     # kR* = k * R* = w * d
        ax.plot(kR, r['spec'], color=color, linestyle=':', lw=2, label='1+1D')

    # 3+1D
    if tag in sh_results:
        s = sh_results[tag]
        d = s['d'] or (bm_results[tag]['d'] if tag in bm_results else 1.)
        kR_sh, y_sh = normalize_sh_spectrum(
            s['k_sq'], s['spec'], d, L=L, zero_pad=pad)
        # k_sq with k=0 stripped (matches y_sh length)
        k_sq_plot = s['k_sq'][1:] if s['k_sq'][0] == 0 else s['k_sq']
        r_sh    = np.sqrt(k_sq_plot)
        N_modes = np.maximum(1., 4 * np.pi * r_sh ** 2)
        yerr    = y_sh * np.sqrt(2. / N_modes)
        ax.plot(kR_sh, y_sh, color=color, linestyle='-', lw=2,
                label=f'3+1D  (t={s["t"]:.1f})')
        ax.fill_between(kR_sh,
                        np.maximum(y_sh - yerr, 1e-300),
                        y_sh + yerr,
                        color=color, alpha=0.2, linewidth=0)
        print(color)

    ax.set_title(rf'$\lambda_\mathrm{{bar}}={lb}$,  $\gamma={gamma:.4g}$')
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel(r'$k R_*$')
    ax.legend(frameon=False, fontsize=9)
    ax.set_ylim(1e1,1e6)

axes[0].set_ylabel(r'$\mathrm{d}E_{\mathrm{GW}}/\mathrm{d}\ln k$')
fig.suptitle('Direct 1+1D vs 3+1D comparison', y=1.01)
plt.tight_layout()
plt.savefig(REPO_ROOT / 'validate_direct.pdf', bbox_inches='tight')
plt.show()